# KNN retrieval testing
Loads the frozen initial KB and evaluates only `unknown_final_test.parquet`. Does not update KB.

In [1]:
from pathlib import Path
import sys, json, shutil, time
import numpy as np
import pandas as pd
import joblib

ROOT=Path.cwd().resolve()
if ROOT.name.lower()=="kb" and ROOT.parent.name.lower()=="notebooks":
    ROOT=ROOT.parent.parent
elif ROOT.name.lower()=="notebooks":
    ROOT=ROOT.parent
KB_DIR=ROOT/"notebooks"/"kb"
sys.path.insert(0,str(KB_DIR))
from kb_flow import read_partition, build, search, atomic_csv, update_after_ae, load_version20, score_precomputed_flow, prepare_ae_context

SPLITS=ROOT/"data/processed/Data_V3_SSH_Unknown_final/unknown_kb_splits"
KB_DATA=ROOT/"data/processed/knn_kb"
MODELS=ROOT/"saved-models/knn_kb"
RESULTS=ROOT/"outputs/results/knn_kb"
for d in (KB_DATA,MODELS,RESULTS): d.mkdir(parents=True,exist_ok=True)
INITIAL_CSV=KB_DATA/"knowledge_base.csv"
INITIAL_INDEX=MODELS/"initial_kb.joblib"
DEMO_CSV=KB_DATA/"knowledge_base_demo.csv"
DEMO_INDEX=MODELS/"demo_kb.joblib"
for name in ("unknown_initial_kb.parquet","unknown_validation.parquet",
             "unknown_final_test.parquet","unknown_demo_kb_updates.parquet"):
    if not (SPLITS/name).is_file(): raise FileNotFoundError(SPLITS/name)
print("Project:",ROOT)
print("Existing splits:",SPLITS)
print("KB outputs:",KB_DATA)

Project: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection
Existing splits: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\Data_V3_SSH_Unknown_final\unknown_kb_splits
KB outputs: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\knn_kb


In [2]:
# Load already trained KB; DO NOT rebuild or tune it with final test.
ref=pd.read_csv(INITIAL_CSV,low_memory=False)
model=joblib.load(INITIAL_INDEX)
if len(ref)!=model["count"]: raise ValueError("KB/index count mismatch")
if model.get("cutoff") is None: raise ValueError("Missing calibrated KB cutoff")
print("Loaded frozen KB:",len(ref),"cutoff:",model["cutoff"])

Loaded frozen KB: 5203 cutoff: 0.9453825241925351


In [3]:
# Frozen label-free record retrieval evaluation (no classification metrics).
test=read_partition(SPLITS/"unknown_final_test.parquet")
start=time.perf_counter()
test_hits=search(test,model,ref,k=5)
elapsed=time.perf_counter()-start
rows=[]
for i,h in enumerate(test_hits):
    rows.append({
        "test_row":i,"matched":h["matched"],
        "nearest_distance":h["nearest_distance"],
        "nearest_pattern_id":h["neighbors"][0]["pattern_id"],
        "top5_pattern_ids":json.dumps([v["pattern_id"] for v in h["neighbors"]]),
        "top5_distances":json.dumps([v["distance"] for v in h["neighbors"]])
    })
test_report=pd.DataFrame(rows)
test_report.to_csv(RESULTS/"frozen_retrieval.csv",index=False)
metrics={"test_rows":len(test_report),
         "match_rate":float(test_report["matched"].mean()),
         "unmatched_rows":int((~test_report["matched"]).sum()),
         "mean_nearest_distance":float(test_report["nearest_distance"].mean()),
         "median_nearest_distance":float(test_report["nearest_distance"].median()),
         "seconds_per_query":elapsed/len(test_report)}
(RESULTS/"frozen_retrieval_metrics.json").write_text(json.dumps(metrics,indent=2))
print("=== FROZEN LABEL-FREE KNN RETRIEVAL TEST ===")
print(json.dumps(metrics,indent=2))


=== FROZEN LABEL-FREE KNN RETRIEVAL TEST ===
{
  "test_rows": 1301,
  "match_rate": 0.9554189085318986,
  "unmatched_rows": 58,
  "mean_nearest_distance": 0.3405475958004121,
  "median_nearest_distance": 0.13477602461101112,
  "seconds_per_query": 0.0019369084550381684
}
